In [8]:
import pandas as pd

orders = pd.read_csv("data/orders.csv")
op_prior = pd.read_csv("data/order_products__prior.csv")
op_train = pd.read_csv("data/order_products__train.csv")
products = pd.read_csv("data/products.csv")
aisles = pd.read_csv("data/aisles.csv")
departments = pd.read_csv("data/departments.csv")

dim_products = (
    products
    .merge(aisles, on="aisle_id", how="left")
    .merge(departments, on="department_id", how="left")
)

In [9]:
# Cell 11 — build a leakage-safe prior-order fact table for feature engineering

prior_orders = orders.loc[
    orders["eval_set"] == "prior",
    [
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
    ],
].copy()

train_orders = orders.loc[
    orders["eval_set"] == "train",
    [
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
    ],
].copy()

prior_order_items = (
    op_prior
    .merge(prior_orders, on="order_id", how="left", validate="many_to_one")
    .merge(
        dim_products[["product_id", "aisle_id", "department_id", "aisle", "department"]],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
)

prior_order_items["order_size"] = prior_order_items.groupby("order_id")["product_id"].transform("size")
prior_order_items["cart_share"] = prior_order_items["add_to_cart_order"] / prior_order_items["order_size"]

prior_order_items.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,aisle_id,department_id,aisle,department,order_size,cart_share
0,2,33120,1,1,202279,3,5,9,8.0,86,16,eggs,dairy eggs,9,0.111111
1,2,28985,2,1,202279,3,5,9,8.0,83,4,fresh vegetables,produce,9,0.222222
2,2,9327,3,0,202279,3,5,9,8.0,104,13,spices seasonings,pantry,9,0.333333
3,2,45918,4,1,202279,3,5,9,8.0,19,13,oils vinegars,pantry,9,0.444444
4,2,30035,5,0,202279,3,5,9,8.0,17,13,baking ingredients,pantry,9,0.555556


In [10]:
# Cell 12 — user-level behavioral features

user_order_features = prior_orders.groupby("user_id", as_index=False).agg(
    user_total_orders=("order_number", "max"),
    user_avg_days_between_orders=("days_since_prior_order", "mean"),
    user_median_days_between_orders=("days_since_prior_order", "median"),
    user_avg_order_hour=("order_hour_of_day", "mean"),
    user_distinct_order_days=("order_dow", "nunique"),
)

user_item_features = prior_order_items.groupby("user_id", as_index=False).agg(
    user_total_items=("product_id", "size"),
    user_distinct_products=("product_id", "nunique"),
    user_reordered_items=("reordered", "sum"),
    user_avg_basket_size=("order_size", "mean"),
    user_avg_cart_share=("cart_share", "mean"),
)

user_hour_pref = (
    prior_orders.groupby(["user_id", "order_hour_of_day"])
    .size()
    .rename("hour_orders")
    .reset_index()
    .sort_values(["user_id", "hour_orders", "order_hour_of_day"], ascending=[True, False, True])
    .drop_duplicates("user_id")
    .rename(columns={"order_hour_of_day": "user_favorite_hour"})[["user_id", "user_favorite_hour"]]
)

user_features = (
    user_order_features
    .merge(user_item_features, on="user_id", how="left", validate="one_to_one")
    .merge(user_hour_pref, on="user_id", how="left", validate="one_to_one")
)

user_features["user_reorder_rate"] = user_features["user_reordered_items"] / user_features["user_total_items"]
user_features["user_product_diversity"] = user_features["user_distinct_products"] / user_features["user_total_items"]

user_features.head()

,user_id,user_total_orders,user_avg_days_between_orders,user_median_days_between_orders,user_avg_order_hour,user_distinct_order_days,user_total_items,user_distinct_products,user_reordered_items,user_avg_basket_size,user_avg_cart_share,user_favorite_hour,user_reorder_rate,user_product_diversity
0,1,10,19.555556,20.0,10.300000,4,59,18,41,6.254237,0.584746,7,0.694915,0.305085
1,2,14,15.230769,13.0,10.571429,5,195,102,93,16.107692,0.535897,10,0.476923,0.523077
2,3,12,12.090909,11.0,16.416667,4,88,33,55,7.886364,0.568182,16,0.625000,0.375000
3,4,5,13.750000,17.0,12.600000,3,18,17,1,4.555556,0.638889,11,0.055556,0.944444
4,5,4,13.333333,11.0,16.000000,3,37,23,14,10.027027,0.554054,18,0.378378,0.621622


In [11]:
# Cell 13 — product-level demand and reorder features

n_users = prior_orders["user_id"].nunique()

product_features = (
    prior_order_items.groupby("product_id", as_index=False)
    .agg(
        product_orders=("product_id", "size"),
        product_users=("user_id", "nunique"),
        product_reordered=("reordered", "sum"),
        product_avg_cart_position=("add_to_cart_order", "mean"),
        product_avg_cart_share=("cart_share", "mean"),
        product_order_span=("order_number", "nunique"),
    )
    .merge(dim_products, on="product_id", how="left", validate="many_to_one")
)

product_features["product_reorder_rate"] = product_features["product_reordered"] / product_features["product_orders"]
product_features["product_user_penetration"] = product_features["product_users"] / n_users

product_features.head()

,product_id,product_orders,product_users,product_reordered,product_avg_cart_position,product_avg_cart_share,product_order_span,product_name,aisle_id,department_id,aisle,department,product_reorder_rate,product_user_penetration
0,1,1852,716,1136,5.801836,0.651364,94,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks,0.613391,0.003472
1,2,90,78,12,9.888889,0.648740,45,All-Seasons Salt,104,13,spices seasonings,pantry,0.133333,0.000378
2,3,277,74,203,6.415162,0.491869,63,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages,0.732852,0.000359
3,4,329,182,147,9.507599,0.567065,40,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen,0.446809,0.000883
4,5,15,6,9,6.466667,0.589401,12,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry,0.600000,0.000029


In [12]:
# Cell 14 — user preference features by department and aisle

user_department_features = prior_order_items.groupby(["user_id", "department_id"], as_index=False).agg(
    user_department_orders=("product_id", "size"),
    user_department_reorders=("reordered", "sum"),
)
user_department_totals = user_department_features.groupby("user_id")["user_department_orders"].transform("sum")
user_department_features["user_department_share"] = user_department_features["user_department_orders"] / user_department_totals
user_department_features["user_department_reorder_rate"] = (
    user_department_features["user_department_reorders"] / user_department_features["user_department_orders"]
)

user_aisle_features = prior_order_items.groupby(["user_id", "aisle_id"], as_index=False).agg(
    user_aisle_orders=("product_id", "size"),
    user_aisle_reorders=("reordered", "sum"),
)
user_aisle_totals = user_aisle_features.groupby("user_id")["user_aisle_orders"].transform("sum")
user_aisle_features["user_aisle_share"] = user_aisle_features["user_aisle_orders"] / user_aisle_totals
user_aisle_features["user_aisle_reorder_rate"] = (
    user_aisle_features["user_aisle_reorders"] / user_aisle_features["user_aisle_orders"]
)

user_department_features.head()

,user_id,department_id,user_department_orders,user_department_reorders,user_department_share,user_department_reorder_rate
0,1,4,5,1,0.084746,0.200000
1,1,7,13,11,0.220339,0.846154
2,1,13,1,0,0.016949,0.000000
3,1,14,3,2,0.050847,0.666667
4,1,16,13,8,0.220339,0.615385


In [13]:
# Cell 15 — user-product interaction features and candidate pairs

user_product_features = (
    prior_order_items.groupby(["user_id", "product_id"], as_index=False)
    .agg(
        up_orders=("product_id", "size"),
        up_reorders=("reordered", "sum"),
        up_first_order_number=("order_number", "min"),
        up_last_order_number=("order_number", "max"),
        up_avg_cart_position=("add_to_cart_order", "mean"),
        up_avg_cart_share=("cart_share", "mean"),
        up_avg_days_since_prior=("days_since_prior_order", "mean"),
    )
    .merge(
        user_features[["user_id", "user_total_orders"]],
        on="user_id",
        how="left",
        validate="many_to_one",
    )
)

user_product_features["up_order_rate"] = user_product_features["up_orders"] / user_product_features["user_total_orders"]
user_product_features["up_reorder_rate"] = user_product_features["up_reorders"] / user_product_features["up_orders"]
user_product_features["up_orders_since_first"] = (
    user_product_features["user_total_orders"] - user_product_features["up_first_order_number"] + 1
)
user_product_features["up_order_rate_since_first"] = (
    user_product_features["up_orders"] / user_product_features["up_orders_since_first"]
)

candidate_pairs = user_product_features[["user_id", "product_id"]].copy()
user_product_features = user_product_features.drop(columns=["user_total_orders"])

user_product_features.head()

,user_id,product_id,up_orders,up_reorders,up_first_order_number,up_last_order_number,up_avg_cart_position,up_avg_cart_share,up_avg_days_since_prior,up_order_rate,up_reorder_rate,up_orders_since_first,up_order_rate_since_first
0,1,196,10,9,1,10,1.400000,0.245278,19.555556,1.0,0.900000,10,1.000000
1,1,10258,9,8,2,10,3.333333,0.562037,19.555556,0.9,0.888889,9,1.000000
2,1,10326,1,0,5,5,5.000000,0.625000,28.000000,0.1,0.000000,6,0.166667
3,1,12427,10,9,1,10,3.300000,0.541667,19.555556,1.0,0.900000,10,1.000000
4,1,13032,3,2,2,10,6.333333,0.962963,21.666667,0.3,0.666667,9,0.333333


In [14]:
# Cell 16 — assemble a model-ready train matrix and validate coverage

train_labels = (
    train_orders[["order_id", "user_id", "order_number"]]
    .merge(candidate_pairs, on="user_id", how="left", validate="one_to_many")
    .merge(
        op_train[["order_id", "product_id"]].assign(target=1),
        on=["order_id", "product_id"],
        how="left",
    )
)

train_labels["target"] = train_labels["target"].fillna(0).astype("int8")

train_matrix = (
    train_labels
    .merge(user_features, on="user_id", how="left", validate="many_to_one")
    .merge(product_features, on="product_id", how="left", validate="many_to_one")
    .merge(user_product_features, on=["user_id", "product_id"], how="left", validate="one_to_one")
    .merge(
        user_department_features[
            ["user_id", "department_id", "user_department_share", "user_department_reorder_rate"]
        ],
        on=["user_id", "department_id"],
        how="left",
    )
    .merge(
        user_aisle_features[
            ["user_id", "aisle_id", "user_aisle_share", "user_aisle_reorder_rate"]
        ],
        on=["user_id", "aisle_id"],
        how="left",
    )
)

train_matrix["orders_since_last_purchase"] = train_matrix["order_number"] - train_matrix["up_last_order_number"]
train_matrix["up_recency_ratio"] = train_matrix["orders_since_last_purchase"] / train_matrix["user_total_orders"]
train_matrix["up_repeat_share_of_user_items"] = train_matrix["up_orders"] / train_matrix["user_total_items"]

for col in [
    "user_department_share",
    "user_department_reorder_rate",
    "user_aisle_share",
    "user_aisle_reorder_rate",
]:
    train_matrix[col] = train_matrix[col].fillna(0)

positive_pairs = train_orders[["order_id", "user_id"]].merge(
    op_train[["order_id", "product_id"]],
    on="order_id",
    how="inner",
)
positive_coverage = positive_pairs.merge(
    candidate_pairs.assign(in_candidates=1),
    on=["user_id", "product_id"],
    how="left",
)

feature_summary = pd.Series(
    {
        "candidate_rows": len(train_matrix),
        "positive_rate": train_matrix["target"].mean(),
        "candidate_coverage_of_true_positives": positive_coverage["in_candidates"].fillna(0).mean(),
        "feature_columns": train_matrix.drop(columns=["order_id", "user_id", "product_id", "target"]).shape[1],
        "rows_with_any_nulls": int(train_matrix.isna().any(axis=1).sum()),
    },
    name="train_matrix_summary",
)

display(feature_summary)
display(
    train_matrix[
        [
            "order_id",
            "user_id",
            "product_id",
            "target",
            "up_orders",
            "orders_since_last_purchase",
            "up_order_rate",
            "product_reorder_rate",
            "user_department_share",
            "user_aisle_share",
        ]
    ].head(10)
)

candidate_rows                          8.474661e+06
positive_rate                           9.780025e-02
candidate_coverage_of_true_positives    5.985944e-01
feature_columns                         4.500000e+01
rows_with_any_nulls                     5.522180e+05
Name: train_matrix_summary, dtype: float64

,order_id,user_id,product_id,target,up_orders,orders_since_last_purchase,up_order_rate,product_reorder_rate,user_department_share,user_aisle_share
0,1187899,1,196,1,10,1,1.0,0.776480,0.220339,0.220339
1,1187899,1,10258,1,9,1,0.9,0.713772,0.372881,0.152542
2,1187899,1,10326,0,1,6,0.1,0.652009,0.084746,0.084746
3,1187899,1,12427,0,10,1,1.0,0.740735,0.372881,0.203390
4,1187899,1,13032,1,3,1,0.3,0.657158,0.050847,0.050847
5,1187899,1,13176,0,2,6,0.2,0.832555,0.084746,0.084746
6,1187899,1,14084,0,1,10,0.1,0.810982,0.220339,0.033898
7,1187899,1,17122,0,1,6,0.1,0.675576,0.084746,0.084746
8,1187899,1,25133,1,8,1,0.8,0.740155,0.220339,0.135593
9,1187899,1,26088,1,2,9,0.2,0.539041,0.372881,0.203390
